# CRYCHIC 多组差异通讯：downsample 小样例

本教程从一个两组或多组 parent H5AD 开始，按 `condition × subject_id × sample_id × cell_type` 固定种子分层下采样，写出可复用的小 H5AD，然后运行 subject-blocked descriptive cross-fit。默认 parent 是可再生 synthetic fixture；设置 `CRYCHIC_MULTIGROUP_H5AD` 后，同一段下采样代码可用于真实原始计数对象。真实数据会保留每组 subject 支持充分、细胞数最多的若干 cell type。

下采样只减少每个生物学单元中的细胞数，不删除 subject，也不把细胞当作重复。教程 profile 默认最多使用 500 条 LR；可通过 `CRYCHIC_TUTORIAL_MAX_INTERACTIONS=none` 运行完整资源。它用于 API smoke、教程和运行时间预估；正式结果必须回到完整队列，并重新检查每个 fold 的支持。


In [1]:
%%time
from __future__ import annotations

import hashlib
import os
import shutil
import time
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse

import crychic
from crychic.attribution import GainCalibrationSpec, PenaltyTuningSpec
from crychic.design import balanced_contrast
from crychic.resources import (
    GeneNamespace,
    Interaction,
    MappingReport,
    ResourceBundle,
    Species,
    TargetPrior,
)
from crychic.sender import ContrastCommonSenderParameters

SEED = 20260718
NOTEBOOK_STARTED = time.perf_counter()
CONTEXT_KEY = "condition"
CONTEXT_LEVELS = ("control", "dose_low", "dose_high")
CONTEXT_LEVELS_VALUE = os.environ.get("CRYCHIC_MULTIGROUP_CONTEXT_LEVELS")
MAX_CELL_TYPES = int(os.environ.get("CRYCHIC_TUTORIAL_MAX_CELL_TYPES", "3"))
MAX_INTERACTIONS_VALUE = os.environ.get(
    "CRYCHIC_TUTORIAL_MAX_INTERACTIONS", "500"
)
MAX_INTERACTIONS = (
    None
    if MAX_INTERACTIONS_VALUE.strip().lower() == "none"
    else int(MAX_INTERACTIONS_VALUE)
)
MAX_CELLS_PER_STRATUM = int(
    os.environ.get("CRYCHIC_DOWNSAMPLE_CELLS_PER_STRATUM", "4")
)
INPUT_VALUE = os.environ.get("CRYCHIC_MULTIGROUP_H5AD")
USE_SYNTHETIC_PARENT = INPUT_VALUE is None
INPUT_H5AD = Path(INPUT_VALUE or "multigroup_parent.h5ad").expanduser()
DATABASE_ROOT = Path(
    os.environ.get("CRYCHIC_DATABASE_ROOT", "resources")
).expanduser()
LR_RESOURCE = os.environ.get("CRYCHIC_LR_RESOURCE", "cellchat").lower()
OUTPUT_ROOT = Path(
    os.environ.get(
        "CRYCHIC_TUTORIAL_OUTPUT_DIR",
        "tutorial_output/multigroup_downsample",
    )
).expanduser()
N_JOBS = int(os.environ.get("CRYCHIC_TUTORIAL_N_JOBS", "1"))
OVERWRITE_OUTPUT = True

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(
    f"CRYCHIC {crychic.__version__}; seed={SEED}; "
    f"max_cells_per_stratum={MAX_CELLS_PER_STRATUM}; "
    f"max_interactions={MAX_INTERACTIONS}"
)


CRYCHIC 0.0.1.dev56+g87153a38e.d20260716; seed=20260718; max_cells_per_stratum=500; max_interactions=500
CPU times: user 1.51 s, sys: 270 ms, total: 1.78 s
Wall time: 1.8 s


## 1. 构造或读取 parent 数据

真实数据必须在 `layers['counts']` 中保存非负 integer-like counts，并包含 `sample_id`、`subject_id`、`cell_type` 和条件列。默认三组各有 4 个独立 subject、每个 subject 一个 sample、两个 cell type；parent 每个 stratum 有 12 个细胞。


In [2]:
%%time
def make_multigroup_parent(cells_per_stratum: int = 12) -> ad.AnnData:
    rng = np.random.default_rng(SEED)
    controls = tuple(f"H{index:02d}" for index in range(1, 9))
    genes = ("L1", "L2", "R1", "R2", "T1", "T2", "T3", "T4", *controls)
    gene_index = {gene: index for index, gene in enumerate(genes)}
    ligand_means = {
        "control": (4.0, 15.0),
        "dose_low": (9.0, 9.0),
        "dose_high": (17.0, 4.0),
    }
    target_means = {
        "control": (3.0, 4.0, 13.0, 10.0),
        "dose_low": (8.0, 8.0, 8.0, 7.0),
        "dose_high": (15.0, 13.0, 3.0, 4.0),
    }
    rows: list[np.ndarray] = []
    obs_rows: list[dict[str, str]] = []
    obs_names: list[str] = []
    for condition in CONTEXT_LEVELS:
        for subject_index in range(4):
            subject_id = f"{condition}-subject-{subject_index + 1:02d}"
            sample_id = f"{condition}-sample-{subject_index + 1:02d}"
            subject_scale = 0.90 + 0.06 * subject_index
            for cell_type in ("Sender", "Receiver"):
                for cell_index in range(cells_per_stratum):
                    means = np.full(len(genes), 2.0 + 0.1 * subject_index)
                    if cell_type == "Sender":
                        means[gene_index["L1"]] = ligand_means[condition][0]
                        means[gene_index["L2"]] = ligand_means[condition][1]
                        means[gene_index["R1"]] = 0.3
                        means[gene_index["R2"]] = 0.3
                    else:
                        means[gene_index["L1"]] = 0.3
                        means[gene_index["L2"]] = 0.3
                        means[gene_index["R1"]] = 20.0
                        means[gene_index["R2"]] = 18.0
                        for gene, value in zip(
                            ("T1", "T2", "T3", "T4"),
                            target_means[condition],
                            strict=True,
                        ):
                            means[gene_index[gene]] = value
                    rows.append(
                        rng.poisson(means * subject_scale).astype(np.int32)
                    )
                    obs_rows.append(
                        {
                            "sample_id": sample_id,
                            "subject_id": subject_id,
                            "cell_type": cell_type,
                            CONTEXT_KEY: condition,
                        }
                    )
                    obs_names.append(
                        f"{sample_id}-{cell_type}-{cell_index:03d}"
                    )
    counts = sparse.csr_matrix(np.vstack(rows), dtype=np.int32)
    result = ad.AnnData(
        X=sparse.csr_matrix(counts.shape, dtype=np.float64),
        obs=pd.DataFrame(obs_rows, index=obs_names),
        var=pd.DataFrame(index=genes),
    )
    result.layers["counts"] = counts
    return result


if USE_SYNTHETIC_PARENT:
    parent = make_multigroup_parent()
    data_origin = "deterministic_multigroup_parent"
else:
    if not INPUT_H5AD.is_file():
        raise FileNotFoundError(INPUT_H5AD)
    parent = ad.read_h5ad(INPUT_H5AD)
    data_origin = "environment_h5ad"

required_obs = {"sample_id", "subject_id", "cell_type", CONTEXT_KEY}
missing_obs = required_obs.difference(parent.obs.columns)
if missing_obs:
    raise ValueError(f"Missing required obs columns: {sorted(missing_obs)}")
if "counts" not in parent.layers:
    raise ValueError("adata.layers['counts'] is required")

available_contexts = tuple(sorted(set(parent.obs[CONTEXT_KEY].astype(str))))
if not USE_SYNTHETIC_PARENT:
    if CONTEXT_LEVELS_VALUE:
        CONTEXT_LEVELS = tuple(
            value.strip()
            for value in CONTEXT_LEVELS_VALUE.split(",")
            if value.strip()
        )
    else:
        CONTEXT_LEVELS = available_contexts
    if len(CONTEXT_LEVELS) < 2 or len(set(CONTEXT_LEVELS)) != len(CONTEXT_LEVELS):
        raise ValueError("At least two unique context levels are required")
    missing_contexts = sorted(set(CONTEXT_LEVELS).difference(available_contexts))
    if missing_contexts:
        raise ValueError(f"Unknown context levels: {missing_contexts}")
    candidate_obs = parent.obs.loc[
        parent.obs[CONTEXT_KEY].astype(str).isin(CONTEXT_LEVELS)
    ]
    support = (
        candidate_obs[[CONTEXT_KEY, "subject_id", "cell_type"]]
        .drop_duplicates()
        .groupby([CONTEXT_KEY, "cell_type"], observed=True)["subject_id"]
        .nunique()
        .unstack(fill_value=0)
        .reindex(index=CONTEXT_LEVELS, fill_value=0)
    )
    eligible_cell_types = tuple(
        str(value) for value in support.columns[(support >= 4).all(axis=0)]
    )
    if not eligible_cell_types:
        raise ValueError(
            "No cell type has >=4 subjects in every selected context"
        )
    cell_counts = candidate_obs["cell_type"].astype(str).value_counts()
    selected_cell_types = tuple(
        sorted(
            eligible_cell_types,
            key=lambda value: (-int(cell_counts.get(value, 0)), value),
        )[:MAX_CELL_TYPES]
    )
    parent = parent[
        parent.obs[CONTEXT_KEY].astype(str).isin(CONTEXT_LEVELS)
        & parent.obs["cell_type"].astype(str).isin(selected_cell_types)
    ].copy()
    data_origin = "environment_h5ad_context_and_cell_type_subset"
else:
    selected_cell_types = tuple(sorted(set(parent.obs["cell_type"].astype(str))))

print(f"data_origin={data_origin}; parent_shape={parent.shape}")
print(f"contexts={CONTEXT_LEVELS}; cell_types={selected_cell_types}")


data_origin=environment_h5ad_context_and_cell_type_subset; parent_shape=(5297, 32738)
contexts=('Normal', 'Tumor'); cell_types=('Epithelial', 'CD1C')
CPU times: user 1.01 s, sys: 194 ms, total: 1.21 s
Wall time: 1.22 s


## 2. 固定种子分层 downsample

分层键保留每个 subject/sample/condition/cell type 的结构。局部随机种子由全局 seed 和规范化 stratum 标签共同生成，因此结果不依赖 Python hash 随机化或输入行顺序。下采样后立即检查 subject 未丢失、每层细胞上限以及 counts layer。


In [3]:
%%time
STRATA_KEYS = (CONTEXT_KEY, "subject_id", "sample_id", "cell_type")


def stratified_downsample(
    data: ad.AnnData,
    *,
    max_cells_per_stratum: int,
    seed: int,
) -> ad.AnnData:
    if max_cells_per_stratum < 1:
        raise ValueError("max_cells_per_stratum must be >= 1")
    selected: list[int] = []
    groups = data.obs.groupby(
        list(STRATA_KEYS), observed=True, sort=True, dropna=False
    ).indices
    for raw_key, raw_positions in groups.items():
        key = raw_key if isinstance(raw_key, tuple) else (raw_key,)
        token = "|".join([str(seed), *(str(value) for value in key)])
        local_seed = int.from_bytes(
            hashlib.sha256(token.encode("utf-8")).digest()[:8],
            byteorder="little",
            signed=False,
        )
        positions = np.asarray(raw_positions, dtype=np.int64)
        if len(positions) > max_cells_per_stratum:
            positions = np.random.default_rng(local_seed).choice(
                positions,
                size=max_cells_per_stratum,
                replace=False,
            )
        selected.extend(int(value) for value in positions)
    return data[np.asarray(sorted(selected), dtype=np.int64)].copy()


adata_small = stratified_downsample(
    parent,
    max_cells_per_stratum=MAX_CELLS_PER_STRATUM,
    seed=SEED,
)
parent_subjects = set(parent.obs["subject_id"].astype(str))
small_subjects = set(adata_small.obs["subject_id"].astype(str))
cells_per_stratum = (
    adata_small.obs.groupby(
        list(STRATA_KEYS), observed=True, dropna=False
    )
    .size()
    .rename("n_cells")
)
assert small_subjects == parent_subjects
assert int(cells_per_stratum.max()) <= MAX_CELLS_PER_STRATUM
assert "counts" in adata_small.layers

small_h5ad = OUTPUT_ROOT / "multigroup_downsample_example.h5ad"
adata_small.write_h5ad(small_h5ad, compression="gzip")
adata = ad.read_h5ad(small_h5ad)
downsample_summary = pd.DataFrame(
    {
        "dataset": ["parent", "downsample"],
        "cells": [parent.n_obs, adata.n_obs],
        "genes": [parent.n_vars, adata.n_vars],
        "subjects": [
            parent.obs["subject_id"].nunique(),
            adata.obs["subject_id"].nunique(),
        ],
        "cell_types": [
            parent.obs["cell_type"].nunique(),
            adata.obs["cell_type"].nunique(),
        ],
    }
)
print(f"small_h5ad={small_h5ad.as_posix()}")
downsample_summary


small_h5ad=../../benchmark_work/tutorial_runs/20260719_021000/multigroup/multigroup_downsample_example.h5ad
CPU times: user 5.69 s, sys: 333 ms, total: 6.03 s
Wall time: 6.27 s


,dataset,cells,genes,subjects,cell_types
0,parent,5297,32738,8,2
1,downsample,5297,32738,8,2


In [4]:
%%time
support_by_group = (
    adata.obs[[CONTEXT_KEY, "subject_id", "cell_type"]]
    .drop_duplicates()
    .groupby([CONTEXT_KEY, "cell_type"], observed=True)["subject_id"]
    .nunique()
    .rename("n_subjects")
    .reset_index()
)
observed_levels = tuple(
    level for level in CONTEXT_LEVELS if level in set(adata.obs[CONTEXT_KEY])
)
if len(observed_levels) < 2:
    raise ValueError("At least two declared context levels are required")
if (support_by_group["n_subjects"] < 4).any():
    raise ValueError(
        "This tiny 2-fold tutorial requires >=4 subjects per condition/cell type"
    )
support_by_group


CPU times: user 5.82 ms, sys: 1.87 ms, total: 7.7 ms
Wall time: 6.69 ms


,condition,cell_type,n_subjects
0,Normal,CD1C,8
1,Normal,Epithelial,8
2,Tumor,CD1C,8
3,Tumor,Epithelial,8


## 3. LR、target prior 与显式 contrasts

Synthetic LR/prior 只用于教程。真实 H5AD 必须配套 checksum-pinned CellChatDB/CellPhoneDB 和 NicheNet prior。这里为全部观测组别预注册两两比较；多组分析不是先看结果再挑 contrast。


In [5]:
%%time
def tutorial_interaction(
    interaction_id: str, ligand: str, receptor: str
) -> Interaction:
    return Interaction(
        interaction_id=interaction_id,
        source_interaction_id=interaction_id,
        ligand_name=ligand,
        receptor_name=receptor,
        ligand_subunits=(ligand,),
        receptor_subunits=(receptor,),
        ligand_is_complex=False,
        receptor_is_complex=False,
        direction="Ligand-Receptor",
        source="synthetic_multigroup_tutorial",
        version="1",
        species=Species.HUMAN,
        gene_namespace=GeneNamespace.HGNC_SYMBOL,
        annotation="Synthetic tutorial contract; not a biological reference",
    )


def tutorial_resources() -> tuple[ResourceBundle, TargetPrior]:
    interactions = (
        tutorial_interaction("tutorial_i1", "L1", "R1"),
        tutorial_interaction("tutorial_i2", "L2", "R2"),
    )
    bundle = ResourceBundle(
        resource_id="crychic_multigroup_tutorial_lr",
        version="1",
        species=Species.HUMAN,
        gene_namespace=GeneNamespace.HGNC_SYMBOL,
        interactions=interactions,
        mapping_report=MappingReport(
            source_rows=2, loaded_rows=2, mapped_entities=4
        ),
        manifest_digest=hashlib.sha256(
            b"crychic-multigroup-tutorial-lr-v1"
        ).hexdigest(),
        source_files=("embedded_synthetic_multigroup_lr",),
        license="CC0-1.0",
        citation="Synthetic CRYCHIC tutorial fixture; not a biological reference.",
    )
    prior = TargetPrior(
        resource_id="crychic_multigroup_tutorial_prior",
        version="1",
        species=Species.HUMAN,
        gene_namespace=GeneNamespace.HGNC_SYMBOL,
        driver_kind="interaction",
        target_ids=("T1", "T2", "T3", "T4"),
        driver_ids=("tutorial_i1", "tutorial_i2"),
        indptr=(0, 2, 4),
        target_indices=(0, 1, 2, 3),
        weights=(1.0, 0.8, 1.0, 0.8),
        ranks=None,
        direction=1,
        evidence="synthetic_multigroup_tutorial",
        mapping_report=MappingReport(
            source_rows=4, loaded_rows=4, mapped_entities=6
        ),
        manifest_digest=hashlib.sha256(
            b"crychic-multigroup-tutorial-prior-v1"
        ).hexdigest(),
    )
    return bundle, prior


def tutorial_resource_subset(
    bundle: ResourceBundle, *, max_interactions: int | None
) -> ResourceBundle:
    if max_interactions is None or len(bundle.interactions) <= max_interactions:
        return bundle
    if max_interactions < 1:
        raise ValueError("CRYCHIC_TUTORIAL_MAX_INTERACTIONS must be >=1 or none")
    ranked = sorted(
        bundle.interactions,
        key=lambda item: hashlib.sha256(
            f"{SEED}|{item.interaction_id}".encode("utf-8")
        ).hexdigest(),
    )
    selected = tuple(ranked[:max_interactions])
    mapped_genes = {
        gene
        for interaction in selected
        for gene in (*interaction.ligand_subunits, *interaction.receptor_subunits)
    }
    selection_digest = hashlib.sha256(
        (
            bundle.manifest_digest
            + "|"
            + "|".join(sorted(item.interaction_id for item in selected))
        ).encode("utf-8")
    ).hexdigest()
    return ResourceBundle(
        resource_id=f"{bundle.resource_id}_tutorial_subset_{max_interactions}",
        version=bundle.version,
        species=bundle.species,
        gene_namespace=bundle.gene_namespace,
        interactions=selected,
        mapping_report=MappingReport(
            source_rows=bundle.mapping_report.source_rows,
            loaded_rows=len(selected),
            mapped_entities=len(mapped_genes),
            notes=(
                *bundle.mapping_report.notes,
                f"tutorial_seeded_hash_subset={max_interactions}",
            ),
        ),
        manifest_digest=selection_digest,
        source_files=bundle.source_files,
        license=bundle.license,
        citation=bundle.citation,
    )


if USE_SYNTHETIC_PARENT:
    lr_resource, target_prior = tutorial_resources()
    resource_origin = "embedded_synthetic_contracts"
else:
    if LR_RESOURCE == "cellchat":
        lr_resource = crychic.load_cellchat_resource(
            DATABASE_ROOT, Species.HUMAN
        )
    elif LR_RESOURCE == "cellphonedb":
        lr_resource = crychic.load_cellphonedb_resource(DATABASE_ROOT)
    else:
        raise ValueError(
            "CRYCHIC_LR_RESOURCE must be cellchat or cellphonedb"
        )
    target_prior = crychic.load_nichenet_target_prior(DATABASE_ROOT)
    full_interaction_count = len(lr_resource.interactions)
    lr_resource = tutorial_resource_subset(
        lr_resource, max_interactions=MAX_INTERACTIONS
    )
    subset_label = (
        "full"
        if len(lr_resource.interactions) == full_interaction_count
        else f"seeded_subset_{len(lr_resource.interactions)}"
    )
    resource_origin = (
        f"checksum_pinned_{LR_RESOURCE}_{subset_label}_and_nichenet"
    )

contrasts = tuple(
    balanced_contrast(
        positive=(observed_levels[positive_index],),
        negative=(observed_levels[negative_index],),
        name=(
            f"{observed_levels[positive_index]}_vs_"
            f"{observed_levels[negative_index]}"
        ),
    )
    for positive_index in range(1, len(observed_levels))
    for negative_index in range(positive_index)
)
contrast_plan = pd.DataFrame(
    [
        {
            "contrast": contrast.name,
            "weights": ", ".join(
                f"{context}:{weight:+.1f}"
                for context, weight in contrast.weights.items()
            ),
        }
        for contrast in contrasts
    ]
)
print(
    f"resource_origin={resource_origin}; contrasts={len(contrasts)}"
)
contrast_plan


resource_origin=checksum_pinned_cellchat_seeded_subset_500_and_nichenet; contrasts=1
CPU times: user 3.34 s, sys: 64.9 ms, total: 3.4 s
Wall time: 3.4 s


,contrast,weights
0,Tumor_vs_Normal,"Normal:-1.0, Tumor:+1.0"


In [6]:
%%time
config = crychic.CrychicConfig(
    context_keys=(CONTEXT_KEY,),
    counts_layer="counts",
    sample_key="sample_id",
    subject_key="subject_id",
    cell_type_key="cell_type",
    design="~ condition",
    random_seed=SEED,
)
def build_pairwise_spec(
    contrast: object, *, seed: int
) -> crychic.CrossFitSpec:
    tuning = PenaltyTuningSpec(
        lambda1_fractions=(1.0,),
        lambda2_fractions=(0.0,),
        inner_allowed_n_splits=(2,),
        min_inner_train_subjects_per_context=1,
        min_inner_validation_subjects_per_context=1,
        root_seed=seed,
    )
    return crychic.CrossFitSpec(
        contrasts=(contrast,),
        outer_fold_partition_seed=seed,
        training_spec=crychic.FoldTrainingSpec(
            min_cells=2,
            min_pooled_availability=0.0,
            max_interactions=None,
            sender_parameters=ContrastCommonSenderParameters(
                min_subjects=2
            ),
        ),
        allowed_n_splits=(2,),
        min_train_subjects_per_context=2,
        min_test_subjects_per_context=1,
        latent_nuisance_spec=crychic.FrozenLatentNuisanceSpec(
            max_components=1,
            min_control_features=4,
            min_training_subjects=2,
            minimum_explained_fraction=0.01,
        ),
        penalty_tuning_spec=tuning,
        gain_calibration_spec=GainCalibrationSpec(
            min_inner_folds=2,
            min_subjects=4,
            min_supported_families=1,
            min_subjects_per_family=2,
            min_positive_observations=4,
            min_distinct_positive_gains=2,
        ),
    )


crossfit_specs = {
    contrast.name: build_pairwise_spec(
        contrast, seed=SEED + index
    )
    for index, contrast in enumerate(contrasts)
}
model = crychic.Crychic(
    config,
    resource_bundle=lr_resource,
    target_prior=target_prior,
)
validated = model.validate(adata)
assert validated.report.n_subjects == len(parent_subjects)
pd.DataFrame(
    [
        {
            "contrast": name,
            "spec_id": spec.spec_id,
            "outer_seed": spec.outer_fold_partition_seed,
        }
        for name, spec in crossfit_specs.items()
    ]
)


CPU times: user 35.4 ms, sys: 28 ms, total: 63.4 ms
Wall time: 62.8 ms


,contrast,spec_id,outer_seed
0,Tumor_vs_Normal,subject_crossfit_spec_35965123e59a0fcf2acb70a0...,20260718


## 4. 运行 descriptive cross-fit

当前稳定主分支按一个 pairwise estimand 写一个结果 bundle，因此这里依次对子集化后的两组完整 subject 运行 3 个预注册 comparisons，再按 contrast 读取拼接。`n_jobs` 只并行每次运行的外层 folds；每个 worker 保留独立 fold-local 状态。小样例默认 1，正式分析可在内存允许时设置 `CRYCHIC_TUTORIAL_N_JOBS`，但不要超过实际 outer fold 数。


In [7]:
%%time
result_root = OUTPUT_ROOT / "crossfit_results"
if OVERWRITE_OUTPUT and result_root.exists():
    shutil.rmtree(result_root)
result_root.mkdir(parents=True, exist_ok=True)

results: dict[str, crychic.CrossFitResult] = {}
run_records: list[dict[str, object]] = []
for contrast in contrasts:
    contrast_name = contrast.name
    contexts = set(contrast.weights)
    pairwise_input = adata[
        adata.obs[CONTEXT_KEY].astype(str).isin(contexts)
    ].copy()
    result_dir = result_root / contrast_name
    started = time.perf_counter()
    result = model.fit_descriptive(
        pairwise_input,
        spec=crossfit_specs[contrast_name],
        n_jobs=N_JOBS,
        output_dir=result_dir,
    )
    elapsed = time.perf_counter() - started
    assert isinstance(result, crychic.CrossFitResult)
    result = crychic.CrossFitResult.load(result_dir)
    results[contrast_name] = result
    run_records.append(
        {
            "contrast": contrast_name,
            "cells": pairwise_input.n_obs,
            "subjects": pairwise_input.obs["subject_id"].nunique(),
            "elapsed_seconds": elapsed,
            "crossfit_result_id": result.manifest[
                "crossfit_result_id"
            ],
            "oof_certified": result.manifest[
                "complete_pipeline_oof_certified"
            ],
            "formal_inference_status": result.manifest[
                "formal_inference_status"
            ],
        }
    )
    completed = len(run_records)
    mean_elapsed = float(
        np.mean([record["elapsed_seconds"] for record in run_records])
    )
    print(
        f"[{completed}/{len(contrasts)}] {contrast_name} complete; "
        f"elapsed={elapsed:.1f}s; "
        f"ETA~{mean_elapsed * (len(contrasts) - completed):.1f}s"
    )

run_summary = pd.DataFrame(run_records)
run_summary


[1/1] Tumor_vs_Normal complete; elapsed=1434.6s; ETA~0.0s
CPU times: user 25min 4s, sys: 22 s, total: 25min 26s
Wall time: 25min 25s


,contrast,cells,subjects,elapsed_seconds,crossfit_result_id,oof_certified,formal_inference_status
0,Tumor_vs_Normal,5297,8,1434.626188,crossfit_result_b62d62172a20d0551da0fa583f21c520,False,not_available_descriptive_only


## 5. 审核状态和差异结果

所有结果按 receiver、contrast 和 status 解释。`observed`、`structural_zero`、`not_estimable` 是不同状态；结构性缺失不能补零。当前结果是 held-out descriptive ledger，不发布正式 p/q、置信区间或 communication probability。


In [8]:
%%time
components = pd.concat(
    [
        result.read_components().assign(result_contrast=name)
        for name, result in results.items()
    ],
    ignore_index=True,
)
differential = pd.concat(
    [result.read_descriptive_differential() for result in results.values()],
    ignore_index=True,
)
receiver_support = pd.concat(
    [
        result.read_receiver_training_support().assign(
            result_contrast=name
        )
        for name, result in results.items()
    ],
    ignore_index=True,
)
forbidden_inference_fields = {
    "p",
    "p_value",
    "q",
    "q_value",
    "fdr",
    "posterior",
    "probability",
    "comm_probability",
    "confidence_interval",
    "standard_error",
}
assert forbidden_inference_fields.isdisjoint(components.columns)
assert forbidden_inference_fields.isdisjoint(differential.columns)
assert set(differential["contrast"].astype(str)) == {
    contrast.name for contrast in contrasts
}
assert not receiver_support.empty

differential_status = (
    differential.groupby(
        ["contrast", "status", "reason_code"],
        observed=True,
        dropna=False,
    )
    .size()
    .rename("rows")
    .reset_index()
)
print(
    f"outer_folds={receiver_support['fold_id'].nunique()}; "
    f"component_rows={len(components)}; "
    f"differential_rows={len(differential)}"
)
differential_status


outer_folds=2; component_rows=456128; differential_rows=19216
CPU times: user 25.4 s, sys: 296 ms, total: 25.7 s
Wall time: 25.7 s


,contrast,status,reason_code,rows
0,Tumor_vs_Normal,observed,NaN,740
1,Tumor_vs_Normal,structural_zero,receptor_family_ineligible,18476


In [9]:
%%time
selected_contrast = contrasts[-1].name
selected_result = results[selected_contrast]
family_view = selected_result.query_family_scores(
    contrast=selected_contrast, mode="state"
)
sender_lr_view = selected_result.query_sender_lr_pairs(
    contrast=selected_contrast, mode="state"
)
observed_effects = differential.loc[
    differential["status"].astype(str).eq("observed"),
    [
        "contrast",
        "receiver",
        "subject_id",
        "family_id",
        "differential_effect",
        "status",
        "reason_code",
    ],
].copy()
print(
    f"contrast={selected_contrast}; family_rows={len(family_view)}; "
    f"sender_lr_rows={len(sender_lr_view)}; "
    f"observed_effect_rows={len(observed_effects)}"
)
observed_effects.head(12)


contrast=Tumor_vs_Normal; family_rows=6912; sender_lr_rows=19200; observed_effect_rows=740
CPU times: user 49.3 s, sys: 242 ms, total: 49.5 s
Wall time: 49.5 s


,contrast,receiver,subject_id,family_id,differential_effect,status,reason_code
76,Tumor_vs_Normal,CD1C,P10,driver_family_0fdd4c2c47cab83179384e514e8ddb33,0.0,observed,NaN
88,Tumor_vs_Normal,CD1C,P10,driver_family_121941eeaf008208a6b7787e7d0e386c,0.0,observed,NaN
104,Tumor_vs_Normal,CD1C,P10,driver_family_16129cc24bd9f325e798b84f7c48365a,0.0,observed,NaN
149,Tumor_vs_Normal,CD1C,P10,driver_family_1fd34e138935fb502afc17377bff34d3,0.0,observed,NaN
160,Tumor_vs_Normal,CD1C,P10,driver_family_227c7e1acd0d4e52923f2fe1e18079bc,0.0,observed,NaN
177,Tumor_vs_Normal,CD1C,P10,driver_family_24e1089426a9d80c24783931e7f42f40,0.0,observed,NaN
282,Tumor_vs_Normal,CD1C,P10,driver_family_3dbf9c59c771508e9be8327e92eb57f7,0.0,observed,NaN
343,Tumor_vs_Normal,CD1C,P10,driver_family_4abb1c1ad930c086be8039e67f4729cc,0.0,observed,NaN
369,Tumor_vs_Normal,CD1C,P10,driver_family_4ee47d28e3b461b15282c348cd1f42f3,0.0,observed,NaN
402,Tumor_vs_Normal,CD1C,P10,driver_family_58503d9e5e47ae54e8d5be5adbe18504,0.0,observed,NaN


In [10]:
%%time
plot_data = observed_effects.loc[
    observed_effects["contrast"].astype(str).eq(selected_contrast)
].copy()
if plot_data.empty:
    print("No observed effects; inspect differential_status and reason_code.")
else:
    summary = (
        plot_data.groupby(["receiver", "family_id"], observed=True)[
            "differential_effect"
        ]
        .mean()
        .sort_values(kind="stable")
        .tail(12)
    )
    labels = [
        f"{receiver} | {family_id[:12]}"
        for receiver, family_id in summary.index
    ]
    fig, ax = plt.subplots(
        figsize=(7.4, max(3.2, 0.42 * len(summary)))
    )
    colors = np.where(summary.to_numpy() >= 0, "#2E6F95", "#C75B39")
    ax.barh(labels, summary.to_numpy(), color=colors)
    ax.axvline(0.0, color="#222222", linewidth=0.8, linestyle="--")
    ax.set_xlabel("Mean held-out subject-family effect (descriptive)")
    ax.set_ylabel("")
    ax.set_title(selected_contrast)
    fig.tight_layout()
    differential_figure = OUTPUT_ROOT / "multigroup_differential.png"
    fig.savefig(differential_figure, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"figure={differential_figure}")
tutorial_elapsed_seconds = time.perf_counter() - NOTEBOOK_STARTED
print(f"tutorial_elapsed_seconds={tutorial_elapsed_seconds:.1f}")


figure=../../benchmark_work/tutorial_runs/20260719_021000/multigroup/multigroup_differential.png
tutorial_elapsed_seconds=1612.3
CPU times: user 126 ms, sys: 22.1 ms, total: 148 ms
Wall time: 158 ms


## 解释边界与正式分析切换

- 本示例是 `k=3`、全部两两比较；真实项目应在看结果前冻结科学问题所需的 contrast，而不是默认穷举后挑选。
- downsample H5AD 可用于代码调试、内存/时间估算和参数合同检查，不能替代全队列结果。正式运行不要设置 interaction cap，并应保留全部合格 subject。
- component/differential 是 receiver-local、held-out descriptive 输出。不要跨 receiver 做无定义的总排名，不要把 `not_estimable` 当零，也不要从当前表声称 p/q、置信区间、communication probability 或因果 sender。
- 若真实数据每组 subject 数不足、某 cell type 在 fold 中缺失，流水线会 fail closed 或保留 typed `not_estimable`；应修正设计/数据支持，而不是静默降低生物学重复要求。
